In [0]:
#need to install fixed version of libraries sa future para di makaissue since laging latest library ang naiinstall kapag hindi nagspecify ng ver

In [0]:
%pip install databricks-vectorsearch mlflow 
dbutils.library.restartPython()

In [0]:
env = dbutils.notebook.run("/Workspace/anihan_tech_np/anihan_tech_np/00_Parameters/env_dependencies", 120)
print(env)

In [0]:
import mlflow.deployments
from databricks.vector_search.client import (
    VectorSearchClient
)

In [0]:
df = spark.sql(f"select appcode, company_name, owner from anihan_tech_{env}.00_master_01_brz.masterdata_company")
display(df)

In [0]:
appcodes = [row["appcode"] for row in df.select("appcode").distinct().collect()]

dbutils.widgets.dropdown(
    "appcode",
    appcodes[0],      
    appcodes,
    "appcode"
)

selected_appcode = dbutils.widgets.get("appcode")
print(selected_appcode)

In [0]:
CATALOG = f"anihan_tech_{env}"
SCHEMA = f"00_{selected_appcode}_01_brz"
VOLUME_PATH = f"/Volumes/anihan_tech_{env}/00_{selected_appcode}_00_lz/lz_vl/pdf"
CHUNK_TABLE = f"{CATALOG}.{SCHEMA}.pdf_chunks"
VECTOR_ENDPOINT = f"{selected_appcode}-pdf-vector-endpoint"
VECTOR_INDEX = f"{CATALOG}.{SCHEMA}.pdf_chunks_index"
EMBEDDING_MODEL = "databricks-bge-large-en"
LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
vsc = VectorSearchClient()

In [0]:
index = vsc.get_index(
    endpoint_name=VECTOR_ENDPOINT,
    index_name=VECTOR_INDEX
)
index.sync()

In [0]:
client = (
    mlflow.deployments
    .get_deploy_client("databricks")
)

In [0]:
def get_embedding(text):

    response = client.predict(
        endpoint=EMBEDDING_MODEL,
        inputs={
            "input": [text]
        }
    )

    return (
        response.data[0]["embedding"]
    )

In [0]:
def retrieve_context(
    question,
    k=5
):
    query_embedding = (
        get_embedding(question)
    )
    results = (
        index.similarity_search(
            query_vector=query_embedding,
            columns=[
                "source",
                "chunk_text"
            ],
            num_results=k
        )
    )

    return results

In [0]:
def build_context(question):
    results = retrieve_context(
        question
    )
    chunks = []
    for row in (
        results["result"]["data_array"]
    ):
        chunks.append(row[1])
    return "\n\n".join(chunks)

In [0]:
def ask_pdf(question):
    context = build_context(
        question
    )
    prompt = f"""
You are an expert research analyst.
Use the provided study content to answer.

Requirements:
- Base answers only on the study.
- Cite findings when possible.
- Summarize complex information.
- Explain implications and recommendations.
- Do not hallucinate information.
- Do not make assumptions.
- Do not make up answers.
- Do not make up names, dates, numbers, organizations, products, or URLs.
- Do not make up sources.

Context:
{context}

Question:
{question}
"""
    response = client.predict(
        endpoint=LLM_MODEL,
        inputs={
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    return (
        response["choices"][0]
        ["message"]["content"]
    )

In [0]:
#test
response = ask_pdf(
    "What is activemos and how does it help in my swine operations?"
)

print(response)